# DR grading: training on Kaggle

Runs the repository's documented pipeline on a free Kaggle GPU. All logic lives
in the `drscreen` package; this notebook only calls `scripts/`.

**Before running**

1. *Settings → Accelerator*: GPU (T4 or P100). *Settings → Internet*: on
   (timm downloads ImageNet weights).
2. *Add Input*: the competition **aptos2019-blindness-detection** (accept its
   rules first) and the dataset
   **aaryapatel98/indian-diabetic-retinopathy-image-dataset**. `kagglehub`
   finds attached inputs, so nothing is downloaded.
3. Set `REPO_URL` below to your copy of this repository.

Outputs land in `/kaggle/working/dr-screening/runs/`. Use *Save Version* to
keep them, then download `best.pt` for the web app.

In [ ]:
REPO_URL = "https://github.com/YOUR-USER/dr-screening.git"
REPO_REF = "main"

In [ ]:
!git clone --depth 1 --branch $REPO_REF $REPO_URL /kaggle/working/dr-screening
%cd /kaggle/working/dr-screening

Kaggle's image already provides torch, torchvision, timm, kagglehub, pandas and
scikit-learn built for its GPU. Installing with `--no-deps` adds our package
without replacing them; the next cell checks that every import resolves.

In [ ]:
!pip install --quiet --no-deps -e .
!python -c "import drscreen.training.trainer, torch; print('CUDA:', torch.cuda.is_available())"

## Data: manifests, cropped cache, inspection
EyeQ is skipped here; the quality model has its own notebook section in a later stage.

In [ ]:
SKIP_EYEQ = "--override sources.eyeq.enabled=false"
!python scripts/fetch_data.py $SKIP_EYEQ
!python scripts/prepare_data.py $SKIP_EYEQ
!python scripts/inspect_data.py $SKIP_EYEQ

In [ ]:
from IPython.display import Image, display

for name in ("aptos_processed_samples.png", "idrid_processed_samples.png"):
    display(Image(f"data/reports/{name}"))

## Train the grading model on APTOS

In [ ]:
!python scripts/train.py --config configs/train_grading.yaml

## External validation on IDRiD
IDRiD was never used for training or model selection.

In [ ]:
from pathlib import Path

run_dir = max(Path("runs").glob("aptos-*"), key=lambda path: path.stat().st_mtime)
checkpoint = run_dir / "best.pt"
print(checkpoint)
!python scripts/evaluate.py --checkpoint {checkpoint}

In [ ]:
import json

report = json.loads((run_dir / "eval-idrid" / "metrics.json").read_text())
for name, value in report["metrics"].items():
    print(f"{name:40s} {value}")
display(Image(str(run_dir / "eval-idrid" / "confusion_matrix.png")))